# Test GCS uploads

This notebook uses the same Application Default Credentials and `.env` settings as the activation-caching script. It uploads a uniquely named text object under `selected_acts/upload_tests/`.

In [ ]:
import os
from datetime import datetime, timezone

import google.auth
from dotenv import find_dotenv, load_dotenv
from google.api_core.exceptions import Forbidden, GoogleAPIError
from google.cloud import storage

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)

PROJECT_ID = os.getenv("GCP_PROJECT_ID", "temporal-interp-exp")
BUCKET_NAME = os.getenv("GCS_BUCKET_NAME", "temporal-research-bucket")
PREFIX = "selected_acts/upload_tests"

print(f".env: {dotenv_path or '<not found>'}")
print(f"Project: {PROJECT_ID}")
print(f"Bucket:  gs://{BUCKET_NAME}")

In [ ]:
# Inspect the credentials that the Python client will actually use.
credentials, detected_project = google.auth.default()
print(f"Credential type: {type(credentials).__name__}")
print(f"Detected project: {detected_project or '<none>'}")
print(f"ADC quota project: {getattr(credentials, 'quota_project_id', None) or '<none>'}")

In [ ]:
# Upload a unique, in-memory test object. No local file is created.
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
object_name = f"{PREFIX}/upload_test_{timestamp}.txt"
gcs_uri = f"gs://{BUCKET_NAME}/{object_name}"
payload = f"GCS upload test at {timestamp}\n"

client = storage.Client(project=PROJECT_ID, credentials=credentials)
bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob(object_name)

try:
    blob.upload_from_string(payload, content_type="text/plain")
except Forbidden as exc:
    print("UPLOAD FAILED: HTTP 403")
    print(exc)
    print("\nLikely remedies:")
    print(f"  1. Grant roles/serviceusage.serviceUsageConsumer on {PROJECT_ID} to this identity.")
    print(f"  2. Grant roles/storage.objectUser on gs://{BUCKET_NAME} to this identity.")
    print(f"  3. Run: gcloud auth application-default set-quota-project {PROJECT_ID}")
    raise
except GoogleAPIError as exc:
    print(f"UPLOAD FAILED: {type(exc).__name__}: {exc}")
    raise
else:
    print(f"UPLOAD SUCCEEDED: {gcs_uri}")
    print(f"Generation: {blob.generation or '<not returned>'}")

In [ ]:
# Optional read-back verification. This requires storage.objects.get.
try:
    downloaded = blob.download_as_text()
except Forbidden as exc:
    print("Upload may have succeeded, but read-back is not permitted for this identity.")
    print(exc)
else:
    assert downloaded == payload
    print(f"READ-BACK SUCCEEDED: {downloaded.strip()}")

In [ ]:
# Set this to True and rerun the cell to remove this notebook's test object.
CLEAN_UP = False

if CLEAN_UP:
    try:
        blob.delete()
    except Forbidden as exc:
        print("Cleanup was not permitted. The test object remains at:", gcs_uri)
        print(exc)
    else:
        print("Deleted:", gcs_uri)
else:
    print("Test object retained at:", gcs_uri)